# Arrays to Model Tensors: A Shape-First Bridge

> **What you already know:** A model receives features, produces predictions, and learns from a loss. You have seen this workflow in Keras.
> **The gap this chapter closes:** Array expressions such as `x[:, -1, :]`, `reshape`, broadcasting, and batched matrix multiplication still feel like syntax tricks instead of visible data movement.
> **What you'll have by the end:** One inspected NumPy data pipeline, the same dense forward pass in NumPy, TensorFlow, and PyTorch, and one text corpus transformed into `(batch, time)` token windows and `(batch, time, embedding)` model inputs.
> **What this chapter is not:** It does not teach backpropagation, RNN internals, attention, or model quality; later chapters own those jobs.

## 0. The job: make every axis earn a name

Riverside House has six manuscript dispatches. Each dispatch has four measured features and one routing label. A second view treats the dispatch text as a token stream for next-token prediction. The same data will force every array operation to answer a practical question.

| Part | Visible problem | Smallest useful operation | Evidence |
|---|---|---|---|
| 1 | A table has numbers but no model contract | Name rows, columns, shape, and dtype | Every axis has a meaning |
| 2 | The model needs subsets, not the whole table | Index and slice rows and columns | Selections retain expected shapes |
| 3 | One record, one batch, and one sequence look deceptively similar | Reshape, transpose, expand, squeeze | Values stay fixed while axes move |
| 4 | Feature scaling invites hidden loops | Broadcast training statistics | Vectorized result matches a loop |
| 5 | Framework APIs can hide the shared computation | Run one dense classifier three ways | NumPy, TensorFlow, and PyTorch logits match |
| 6 | Raw text has unequal-length lines | Encode, window, and batch token IDs | Inputs and shifted targets align |
| 7 | An RNN expects vectors, not IDs | Gather embedding rows | All frameworks produce `(B, T, D)` |

**Predict:** Which will cause more trouble later: forgetting an API name, or silently swapping the batch and time axes?

In [ ]:
# Setup: three array systems, one deterministic experiment
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import tensorflow as tf

SEED = 23
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
torch.manual_seed(SEED)

def inspect(name, value, axes):
    shape = tuple(value.shape)
    dtype = getattr(value, "dtype", type(value).__name__)
    print(f"{name:<24} shape={str(shape):<18} axes={axes:<30} dtype={dtype}")

print(f"NumPy {np.__version__} | TensorFlow {tf.__version__} | PyTorch {torch.__version__}")
print("Rule for this notebook: predict the shape, name every axis, then run the operation.")

---

## Part 1 - A table is already a tensor contract

A spreadsheet encourages you to think in column names. A model receives an array. Once the names disappear, position carries the contract:

```text
axis 0 = dispatch examples
axis 1 = measured features
X.shape = (examples, features)
```

The four features are word count, named-entity count, urgency score, and ambiguity score. The binary target says whether an editor escalated the dispatch.

In [ ]:
# Build a labeled DataFrame, then cross the model-input boundary explicitly
dispatch_table = pd.DataFrame({
    "dispatch": [
        "missing chapter", "clear approval", "rights conflict",
        "routine update", "timeline risk", "copy edit",
    ],
    "word_count": [42, 18, 35, 24, 31, 20],
    "entity_count": [3, 1, 4, 1, 2, 0],
    "urgency": [0.90, 0.15, 0.85, 0.20, 0.75, 0.10],
    "ambiguity": [0.80, 0.10, 0.70, 0.15, 0.55, 0.20],
    "escalated": [1, 0, 1, 0, 1, 0],
})
feature_names = np.array(["word_count", "entity_count", "urgency", "ambiguity"], dtype=str)
dispatch_names = dispatch_table["dispatch"].to_numpy(dtype=str)

# DataFrames retain semantic labels and mixed column types; model arrays retain values, shape, and dtype.
X_raw = dispatch_table[feature_names.tolist()].to_numpy(dtype=np.float32)
y_raw = dispatch_table["escalated"].to_numpy(dtype=np.int64)

print(dispatch_table.head(2).to_string(index=False))
print("\nDataFrame columns:", dispatch_table.columns.tolist())
inspect("X_raw", X_raw, "examples, features")
inspect("y_raw", y_raw, "examples")
assert X_raw.shape == (6, 4)
assert y_raw.shape == (6,)
print("One array row paired back with its saved labels:", dict(zip(feature_names, X_raw[0])))
print("PASS: the DataFrame carried names; the NumPy arrays carry the homogeneous model values.")


### Predict first - what survives an index?

Before running the next cell, predict each shape:

- `X_raw[0]`
- `X_raw[0:1]`
- `X_raw[:, 2]`
- `X_raw[:, 2:3]`
- `X_raw[[0, 2, 4]][:, [0, 2, 3]]`

The difference between an integer index and a one-element slice is load-bearing. An integer removes an axis; a slice preserves it. Models usually expect the batch axis to survive.

In [ ]:
# Indexing and slicing: select examples and features without losing the contract by accident
one_example = X_raw[0]
one_example_batch = X_raw[0:1]
urgency_vector = X_raw[:, 2]
urgency_column = X_raw[:, 2:3]
escalated_feature_subset = X_raw[[0, 2, 4]][:, [0, 2, 3]]

inspect("one_example", one_example, "features")
inspect("one_example_batch", one_example_batch, "batch, features")
inspect("urgency_vector", urgency_vector, "examples")
inspect("urgency_column", urgency_column, "examples, one feature")
inspect("escalated_subset", escalated_feature_subset, "selected examples, selected features")

assert one_example.shape == (4,)
assert one_example_batch.shape == (1, 4)
assert urgency_vector.shape == (6,)
assert urgency_column.shape == (6, 1)
print("PASS: slices preserve axes; integer indices remove them.")

#### What just happened - and why it matters

`X_raw[0]` and `X_raw[0:1]` contain the same four numbers, but they do not make the same promise. `(4,)` says only "features." `(1, 4)` says "one example, four features." A dense layer can often accept both, which makes accidental axis loss especially quiet.

**Your turn:** change `0:1` to `0:2`. Predict the shape and name both axes before running the cell again.

---

## Part 2 - Reshape changes the map, not the values

Suppose six dispatches arrive in three review batches of two. `reshape(3, 2, 4)` does not invent or reorder values; it changes how you address the same 24 numbers.

A transpose is different: it reorders axes. The numbers still do not change, but their coordinates do. This is how a batch-first sequence `(B, T, D)` becomes a head-first attention tensor `(B, H, T, D_h)` later.

In [ ]:
# Reshape, transpose, expand, and squeeze while checking value preservation
review_batches = X_raw.reshape(3, 2, 4)
feature_first = review_batches.transpose(0, 2, 1)
single_batch = np.expand_dims(X_raw[0], axis=0)
round_trip = np.squeeze(single_batch, axis=0)

inspect("review_batches", review_batches, "batches, examples, features")
inspect("feature_first", feature_first, "batches, features, examples")
inspect("single_batch", single_batch, "batch, features")
inspect("round_trip", round_trip, "features")

assert np.array_equal(review_batches.reshape(6, 4), X_raw)
assert np.array_equal(round_trip, X_raw[0])
print("PASS: reshape preserved value order; transpose changed axis order; expand/squeeze added and removed one size-1 axis.")

In [ ]:
# Static shape storyboard: the same 24 values under three useful contracts
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for axis, array, title, labels in [
    (axes[0], X_raw, "Table: (N, F)", ("features", "examples")),
    (axes[1], review_batches.reshape(6, 4), "Batches: (3, 2, F)", ("features", "batch x example")),
    (axes[2], feature_first.reshape(3 * 4, 2), "Transposed: (3, F, 2)", ("examples within batch", "batch x feature")),
]:
    image = axis.imshow(array, cmap="Blues", aspect="auto")
    axis.set_title(title)
    axis.set_xlabel(labels[0])
    axis.set_ylabel(labels[1])
fig.colorbar(image, ax=axes, shrink=0.75, label="stored value")
fig.suptitle("Shape changes how values are addressed; it does not create new evidence")
plt.show()

---

## Part 3 - Broadcasting replaces repeated bookkeeping

The raw columns live on incompatible scales: word count reaches the forties while urgency stays below one. Standardization needs one mean and one standard deviation per feature.

The tempting implementation loops over every row. Broadcasting expresses the real intent: apply the same four training statistics to every example. NumPy compares shapes from the right, so `(6, 4)` and `(4,)` align on the feature axis.

In [ ]:
# Fit feature statistics on training rows, then broadcast across every row
train_indices = np.array([0, 1, 2, 3], dtype=np.int64)
test_indices = np.array([4, 5], dtype=np.int64)
X_train_raw = X_raw[train_indices]
X_test_raw = X_raw[test_indices]
y_train = y_raw[train_indices]
y_test = y_raw[test_indices]

feature_mean = X_train_raw.mean(axis=0)
feature_std = X_train_raw.std(axis=0) + 1e-6
X_train = (X_train_raw - feature_mean) / feature_std
X_test = (X_test_raw - feature_mean) / feature_std

loop_result = np.empty_like(X_train)
for row in range(X_train_raw.shape[0]):
    loop_result[row] = (X_train_raw[row] - feature_mean) / feature_std

inspect("feature_mean", feature_mean, "features")
inspect("X_train", X_train, "batch, features")
inspect("X_test", X_test, "batch, features")
assert np.allclose(X_train, loop_result)
assert np.allclose(X_train.mean(axis=0), 0.0, atol=1e-5)
print("PASS: broadcasting matched the explicit row loop, and test rows used training statistics only.")

#### A useful failure: incompatible trailing dimensions

`(6, 4) + (4,)` works because the trailing dimensions match. `(6, 4) + (6,)` fails because NumPy compares `4` with `6`. Broadcasting is not guessing; it follows a small alignment rule.

**Predict:** What shape should you give six row-specific offsets so one value applies to each row?

In [ ]:
# Repair a broadcasting mismatch by naming the missing feature axis
row_offsets = np.arange(6, dtype=np.float32)
try:
    _ = X_raw + row_offsets
except ValueError as error:
    print("Expected mismatch:", error)

row_offsets_column = row_offsets[:, None]
shifted_rows = X_raw + row_offsets_column
inspect("row_offsets", row_offsets, "examples")
inspect("row_offsets[:, None]", row_offsets_column, "examples, one feature")
inspect("shifted_rows", shifted_rows, "examples, features")
assert shifted_rows.shape == X_raw.shape
print("PASS: adding a size-1 feature axis made the intended row-wise broadcast explicit.")

---

## Part 4 - One classifier, first as visible array math

A dense classifier does not consume a DataFrame. It consumes a floating-point matrix `(batch, features)`, multiplies it by a weight matrix, and adds one bias per output.

```text
X                 @ W                 + b          = logits
(batch, features) @ (features, class) + (class,)   = (batch, class)
```

The bias broadcasts across the batch axis. This is the same rule you just used for normalization.

In [ ]:
# NumPy forward pass for a two-class dispatch router
W_numpy = np.array([
    [-0.30, 0.30],
    [-0.15, 0.15],
    [-0.80, 0.80],
    [-0.60, 0.60],
], dtype=np.float32)
b_numpy = np.array([0.10, -0.10], dtype=np.float32)

logits_numpy = X_train @ W_numpy + b_numpy
predictions_numpy = logits_numpy.argmax(axis=1)

inspect("X_train", X_train, "batch, features")
inspect("W_numpy", W_numpy, "features, classes")
inspect("b_numpy", b_numpy, "classes")
inspect("logits_numpy", logits_numpy, "batch, classes")
print("Predictions:", predictions_numpy.tolist())
assert logits_numpy.shape == (len(train_indices), 2)
print("PASS: matrix multiplication contracted the feature axis; batch and class axes survived.")

---

## Part 5 - TensorFlow tensors: arrays with framework contracts

A TensorFlow tensor still has values, shape, axes, and dtype. Unlike a NumPy array, it can participate in TensorFlow's device execution and automatic differentiation. Unlike a DataFrame, it does not carry semantic column names. You must preserve those meanings yourself.

The operations are familiar:

| NumPy | TensorFlow | Meaning |
|---|---|---|
| `x[:, 2:3]` | `x[:, 2:3]` | Preserve a one-column feature axis |
| `x.reshape(...)` | `tf.reshape(x, ...)` | Reinterpret element layout |
| `x.transpose(...)` | `tf.transpose(x, perm=...)` | Reorder axes |
| `x[:, None]` | `tf.expand_dims(x, axis=1)` | Add a size-1 axis |
| `x @ W` | `tf.linalg.matmul(x, W)` | Contract matching inner axes |

In [ ]:
# Repeat the shape operations as TensorFlow tensors
X_tf = tf.convert_to_tensor(X_raw, dtype=tf.float32)
batch_tf = tf.reshape(X_tf, (3, 2, 4))
feature_first_tf = tf.transpose(batch_tf, perm=(0, 2, 1))
urgency_column_tf = X_tf[:, 2:3]
row_offsets_tf = tf.expand_dims(tf.range(6, dtype=tf.float32), axis=1)
shifted_tf = X_tf + row_offsets_tf

inspect("X_tf", X_tf, "examples, features")
inspect("batch_tf", batch_tf, "batches, examples, features")
inspect("feature_first_tf", feature_first_tf, "batches, features, examples")
inspect("urgency_column_tf", urgency_column_tf, "examples, one feature")

assert np.array_equal(batch_tf.numpy(), review_batches)
assert np.array_equal(feature_first_tf.numpy(), feature_first)
assert np.array_equal(shifted_tf.numpy(), shifted_rows)
print("PASS: TensorFlow changed the API surface, not the array semantics.")

In [ ]:
# Load the NumPy weights into a Keras Dense layer and prove forward-pass parity
dense_tf = tf.keras.layers.Dense(2, use_bias=True)
_ = dense_tf(tf.zeros((1, 4), dtype=tf.float32))
dense_tf.set_weights([W_numpy, b_numpy])
logits_tf = dense_tf(tf.convert_to_tensor(X_train, dtype=tf.float32))

inspect("logits_tf", logits_tf, "batch, classes")
assert np.allclose(logits_tf.numpy(), logits_numpy, atol=1e-6)
print("PASS: Keras Dense and NumPy produced identical logits from identical values.")
print("Keras stores Dense weights as (input_features, output_units).")

#### Tensor does not mean mysterious multidimensional object

A tensor is an array plus execution rules. TensorFlow adds gradient recording, devices, graph tracing, and immutable tensor values. None of those additions changes what axis 0 means.

The dangerous mistake is to treat shape as decoration. Frameworks can only report that dimensions do not align; they cannot tell you that you accidentally interpreted `time` as `features`.

---

## Part 6 - PyTorch tensors: the same axes, more explicit state

PyTorch keeps dtype and device visible, and its common method names differ slightly. The underlying contracts remain the same.

| NumPy | PyTorch | Meaning |
|---|---|---|
| `x.reshape(...)` | `x.reshape(...)` | Reinterpret element layout |
| `x.transpose(0, 2, 1)` | `x.permute(0, 2, 1)` | Reorder several axes |
| `x[:, None]` | `x.unsqueeze(1)` | Add a size-1 axis |
| `np.stack(items)` | `torch.stack(items)` | Create a new axis |
| `np.concatenate(items)` | `torch.cat(items)` | Extend an existing axis |
| `x @ W` | `x @ W` | Contract matching inner axes |

In [ ]:
# Repeat the same shape operations as PyTorch tensors
X_torch = torch.as_tensor(X_raw, dtype=torch.float32)
batch_torch = X_torch.reshape(3, 2, 4)
feature_first_torch = batch_torch.permute(0, 2, 1)
urgency_column_torch = X_torch[:, 2:3]
row_offsets_torch = torch.arange(6, dtype=torch.float32).unsqueeze(1)
shifted_torch = X_torch + row_offsets_torch

inspect("X_torch", X_torch, "examples, features")
inspect("batch_torch", batch_torch, "batches, examples, features")
inspect("feature_first_torch", feature_first_torch, "batches, features, examples")
inspect("urgency_column_torch", urgency_column_torch, "examples, one feature")

assert np.array_equal(batch_torch.numpy(), review_batches)
assert np.array_equal(feature_first_torch.numpy(), feature_first)
assert np.array_equal(shifted_torch.numpy(), shifted_rows)
print("PASS: PyTorch changed the API surface, not the array semantics.")

In [ ]:
# Load the same weights into nn.Linear and prove three-framework parity
dense_torch = nn.Linear(4, 2, bias=True)
with torch.no_grad():
    dense_torch.weight.copy_(torch.from_numpy(W_numpy.T))
    dense_torch.bias.copy_(torch.from_numpy(b_numpy))

logits_torch = dense_torch(torch.from_numpy(X_train))

inspect("dense_torch.weight", dense_torch.weight, "classes, features")
inspect("logits_torch", logits_torch, "batch, classes")
assert np.allclose(logits_torch.detach().numpy(), logits_numpy, atol=1e-6)
assert np.allclose(logits_torch.detach().numpy(), logits_tf.numpy(), atol=1e-6)
print("PASS: NumPy, TensorFlow/Keras, and PyTorch produced identical logits.")
print("PyTorch stores Linear weights as (output_features, input_features), so the NumPy matrix was transposed once when copied.")

### Your turn - find the axis bug before the framework does

The model expects `(batch, features)`. Predict why `X_train.T` cannot enter the same layer even though it contains exactly the same numbers. Then uncomment the failing call, read the error, and repair the input without changing the model.

In [ ]:
# Shape-error drill: change TRY_BROKEN_INPUT to True after making a prediction
TRY_BROKEN_INPUT = False
transposed_input = torch.from_numpy(X_train.T.copy())
inspect("transposed_input", transposed_input, "features, batch")

if TRY_BROKEN_INPUT:
    try:
        dense_torch(transposed_input)
    except RuntimeError as error:
        print("Expected model-boundary failure:", error)

repaired_input = transposed_input.T.contiguous()
repaired_logits = dense_torch(repaired_input)
assert torch.allclose(repaired_logits, logits_torch)
print("PASS: restoring (batch, features) restored the original logits.")

---

## Part 7 - Raw text becomes supervised sequence windows

The classifier used `(batch, features)`. An RNN language model expects token IDs shaped `(batch, time)`, then embedding lookup expands them to `(batch, time, embedding)`.

The raw corpus is deliberately tiny and already whitespace-tokenized. Vocabulary algorithms, padding policy, and BPE belong to prerequisite 06. Here the job is narrower: expose how slicing turns one token stream into aligned training examples.

```text
aria heard the signal aboard meridian

context length = 3
[aria, heard, the]     -> signal
[heard, the, signal]   -> aboard
[the, signal, aboard]  -> meridian
```

In [ ]:
# Encode a tiny corpus as one integer stream
corpus_lines = [
    "aria heard the signal aboard meridian",
    "wren decoded the message aboard meridian",
]
words = [word for line in corpus_lines for word in line.split()]
vocabulary = ["<eos>"] + sorted(set(words))
token_to_id = {token: index for index, token in enumerate(vocabulary)}
id_to_token = {index: token for token, index in token_to_id.items()}

stream_tokens = []
for line in corpus_lines:
    stream_tokens.extend(line.split())
    stream_tokens.append("<eos>")
token_stream = np.array([token_to_id[token] for token in stream_tokens], dtype=np.int64)

inspect("token_stream", token_stream, "time")
print("tokens:", stream_tokens)
print("ids:   ", token_stream.tolist())
assert len(token_stream) == len(stream_tokens)

### Predict first - how many windows fit?

For stream length `L` and context length `T`, the number of next-token examples is `L - T`. Each example needs `T` input IDs and the immediately following target ID.

A Python loop makes the rule obvious. Fancy indexing then applies the same offsets to every starting position at once. Both should produce identical arrays.

In [ ]:
# Build overlapping next-token windows first with a loop, then with broadcasted indices
CONTEXT_LENGTH = 3
window_count = len(token_stream) - CONTEXT_LENGTH

loop_inputs = np.stack([
    token_stream[start:start + CONTEXT_LENGTH]
    for start in range(window_count)
])
loop_targets = np.array([
    token_stream[start + CONTEXT_LENGTH]
    for start in range(window_count)
], dtype=np.int64)

start_indices = np.arange(window_count)[:, None]
time_offsets = np.arange(CONTEXT_LENGTH)[None, :]
window_indices = start_indices + time_offsets
window_inputs = token_stream[window_indices]
window_targets = token_stream[np.arange(CONTEXT_LENGTH, len(token_stream))]

inspect("start_indices", start_indices, "examples, one offset")
inspect("time_offsets", time_offsets, "one example, time")
inspect("window_indices", window_indices, "examples, time")
inspect("window_inputs", window_inputs, "examples, time")
inspect("window_targets", window_targets, "examples")

assert np.array_equal(window_inputs, loop_inputs)
assert np.array_equal(window_targets, loop_targets)
for row in range(3):
    context = [id_to_token[int(token_id)] for token_id in window_inputs[row]]
    target = id_to_token[int(window_targets[row])]
    print(f"{context} -> {target}")
print("PASS: broadcasted index grids reproduced the readable sliding-window loop.")

#### One more reshape creates mini-batches

`window_inputs` currently uses `(examples, time)`. To process two examples together, keep the time axis intact and group the example axis. `reshape(number_of_batches, batch_size, time)` is valid only when the example count divides evenly; otherwise a real pipeline must keep or pad the remainder rather than silently dropping it.

In [ ]:
# Form complete mini-batches without hiding the remainder policy
BATCH_SIZE = 2
complete_example_count = (window_count // BATCH_SIZE) * BATCH_SIZE
remainder_count = window_count - complete_example_count
batched_inputs = window_inputs[:complete_example_count].reshape(-1, BATCH_SIZE, CONTEXT_LENGTH)
batched_targets = window_targets[:complete_example_count].reshape(-1, BATCH_SIZE)

inspect("batched_inputs", batched_inputs, "mini-batches, batch, time")
inspect("batched_targets", batched_targets, "mini-batches, batch")
print(f"Explicit remainder policy: {remainder_count} example(s) left for a smaller final batch.")
assert batched_inputs.shape[1:] == (BATCH_SIZE, CONTEXT_LENGTH)
assert batched_targets.shape[1:] == (BATCH_SIZE,)
print("PASS: reshaping grouped examples without mixing the time axis.")

---

## Part 8 - Token IDs become RNN-ready vectors

An integer token ID is an address, not a numeric feature. Embedding lookup gathers one learned row per ID:

```text
token IDs:   (batch, time)
table:       (vocabulary, embedding)
result:      (batch, time, embedding)
```

The lookup preserves both existing axes and appends the embedding axis. NumPy fancy indexing, `tf.gather`, Keras `Embedding`, PyTorch indexing, and `nn.Embedding` all implement that same contract.

In [ ]:
# Prove embedding lookup parity and the RNN input contract in all three frameworks
EMBEDDING_DIM = 4
embedding_table = np.linspace(
    -0.5, 0.5, num=len(vocabulary) * EMBEDDING_DIM, dtype=np.float32
).reshape(len(vocabulary), EMBEDDING_DIM)
example_batch_ids = batched_inputs[0]

embedded_numpy = embedding_table[example_batch_ids]

ids_tf = tf.convert_to_tensor(example_batch_ids, dtype=tf.int32)
embedded_tf = tf.gather(tf.convert_to_tensor(embedding_table), ids_tf)
embedding_tf_layer = tf.keras.layers.Embedding(len(vocabulary), EMBEDDING_DIM)
embedding_tf_layer.build((None, CONTEXT_LENGTH))
embedding_tf_layer.set_weights([embedding_table])
embedded_tf_layer_output = embedding_tf_layer(ids_tf)

ids_torch = torch.as_tensor(example_batch_ids, dtype=torch.long)
embedded_torch = torch.as_tensor(embedding_table)[ids_torch]
embedding_torch_layer = nn.Embedding(len(vocabulary), EMBEDDING_DIM)
with torch.no_grad():
    embedding_torch_layer.weight.copy_(torch.from_numpy(embedding_table))
embedded_torch_layer_output = embedding_torch_layer(ids_torch)

inspect("example_batch_ids", example_batch_ids, "batch, time")
inspect("embedded_numpy", embedded_numpy, "batch, time, embedding")
inspect("embedded_tf", embedded_tf, "batch, time, embedding")
inspect("embedded_torch", embedded_torch, "batch, time, embedding")

assert np.allclose(embedded_tf.numpy(), embedded_numpy)
assert np.allclose(embedded_tf_layer_output.numpy(), embedded_numpy)
assert np.allclose(embedded_torch.numpy(), embedded_numpy)
assert np.allclose(embedded_torch_layer_output.detach().numpy(), embedded_numpy)
print("PASS: five lookup paths produced the same (batch, time, embedding) values.")

In [ ]:
# Minimal RNN boundary check: consume the prepared vectors without training
HIDDEN_SIZE = 5
keras_rnn = tf.keras.layers.SimpleRNN(HIDDEN_SIZE, return_sequences=True, return_state=True)
keras_sequence, keras_final_state = keras_rnn(embedded_tf)

torch_rnn = nn.RNN(EMBEDDING_DIM, HIDDEN_SIZE, batch_first=True)
torch_sequence, torch_final_state = torch_rnn(embedded_torch)

inspect("keras_sequence", keras_sequence, "batch, time, hidden")
inspect("keras_final_state", keras_final_state, "batch, hidden")
inspect("torch_sequence", torch_sequence, "batch, time, hidden")
inspect("torch_final_state", torch_final_state, "layers, batch, hidden")

assert tuple(keras_sequence.shape) == (BATCH_SIZE, CONTEXT_LENGTH, HIDDEN_SIZE)
assert tuple(torch_sequence.shape) == (BATCH_SIZE, CONTEXT_LENGTH, HIDDEN_SIZE)
assert tuple(keras_final_state.shape) == (BATCH_SIZE, HIDDEN_SIZE)
assert tuple(torch_final_state.shape) == (1, BATCH_SIZE, HIDDEN_SIZE)
print("PASS: both RNNs accepted the same batch-first sequence input.")
print("The extra leading PyTorch state axis names recurrent layer/direction; it is not another batch.")

#### The final reshape serves the loss, not the model

The RNN preserves one hidden vector per token: `(batch, time, hidden)`. A vocabulary projection turns each hidden vector into logits `(batch, time, vocabulary)`. Sparse cross-entropy then treats every token position as one classification example, so both logits and targets flatten across **batch and time together**:

```text
logits   (B, T, V) -> (B x T, V)
targets  (B, T)    -> (B x T)
```

This reshape does not mix vocabulary scores into examples. The vocabulary axis remains intact because it is the class axis scored by cross-entropy. We stop at a real finite loss; the following Antarctic and RNN bridge notebooks own backward passes and optimizer updates.


In [ ]:
# Compute real next-token losses in TensorFlow and PyTorch from the prepared arrays
sequence_targets_numpy = np.roll(example_batch_ids, shift=-1, axis=1)
sequence_targets_numpy[:, -1] = batched_targets[0]

# TensorFlow: RNN output (B, T, H) -> vocabulary logits (B, T, V) -> flattened loss inputs.
tf_projection = tf.keras.layers.Dense(len(vocabulary))
targets_tf = tf.convert_to_tensor(sequence_targets_numpy, dtype=tf.int32)
tf_logits = tf_projection(keras_sequence)
tf_flat_logits = tf.reshape(tf_logits, (-1, len(vocabulary)))
tf_flat_targets = tf.reshape(targets_tf, (-1,))
tf_loss = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(
    tf_flat_targets, tf_flat_logits, from_logits=True
))

# PyTorch: preserve V while flattening B and T into one supervised-example axis.
torch_projection = nn.Linear(HIDDEN_SIZE, len(vocabulary))
targets_torch = torch.as_tensor(sequence_targets_numpy, dtype=torch.long)
torch_logits = torch_projection(torch_sequence)
torch_flat_logits = torch_logits.reshape(-1, len(vocabulary))
torch_flat_targets = targets_torch.reshape(-1)
torch_loss = nn.functional.cross_entropy(torch_flat_logits, torch_flat_targets)

inspect("tf_logits", tf_logits, "batch, time, vocabulary")
inspect("torch_logits", torch_logits, "batch, time, vocabulary")
inspect("tf_flat_logits", tf_flat_logits, "batch x time, vocabulary")
inspect("torch_flat_targets", torch_flat_targets, "batch x time")
assert tuple(tf_logits.shape) == tuple(torch_logits.shape) == (
    BATCH_SIZE, CONTEXT_LENGTH, len(vocabulary)
)
assert tuple(tf_flat_logits.shape) == tuple(torch_flat_logits.shape)
assert torch_flat_targets.numel() == BATCH_SIZE * CONTEXT_LENGTH
assert np.isfinite(tf_loss.numpy()).all() and torch.isfinite(torch_loss).item()
print(f"TensorFlow next-token loss: {tf_loss.numpy().item():.4f}")
print(f"PyTorch next-token loss:    {torch_loss.item():.4f}")
print("PASS: prepared windows reached real next-token losses in both frameworks.")


### Your turn - change one known axis

Set `CONTEXT_LENGTH = 4` in Part 7 and rerun Parts 7-8. Before running, predict:

1. How many windows remain?
2. Which axis changes in `(examples, time)`?
3. Which axis changes after embedding in `(batch, time, embedding)`?
4. Which axis does **not** change?

The answer should come from the axis meanings, not from memorizing one tuple.

---

## What This Chapter Covered (and What It Didn't)

| Coverage tier | Topics | Evidence |
|---|---|---|
| Built and measured | Axis naming; indexing; slicing; reshape; transpose/permute; expand/squeeze; broadcasting; matrix multiplication; text windows; batching; embedding lookup | Shape assertions, loop/vectorized parity, three-framework logit parity, five-path embedding parity |
| Explained or illustrated | Views versus semantic axis meaning; training-only normalization statistics; incomplete final batches; framework tensor contracts | Static shape storyboard and explicit failure demonstrations |
| Named and out of scope | Gradients, optimizers, `Dataset`/`DataLoader`, padding masks, BPTT, attention | Owned by the Antarctic field guide, tokenization prerequisite, RNN bridge, and Transformer foundations |

### Scorecard

| Opening question | Answer now backed by evidence |
|---|---|
| What does an axis mean? | It names an independent way to address the data: example, feature, time, class, or embedding coordinate. |
| Does reshape change values? | No; the round-trip assertion recovered the original table exactly. |
| Why does broadcasting work? | Trailing dimensions match or one side has size 1; the vectorized normalization matched the explicit loop. |
| Are TensorFlow and PyTorch tensors new mathematics? | No; both reproduced NumPy indexing, shape transforms, and classifier logits. |
| How does raw text reach an RNN? | Tokens become IDs `(B,T)`, embedding lookup appends `D`, and the RNN consumes `(B,T,D)`. |

### Key insights to keep

- A shape is a contract: always name each axis.
- Integer indexing removes an axis; slicing can preserve it.
- Reshape groups existing values; transpose changes axis order.
- Broadcasting aligns from the trailing dimensions and never guesses your intent.
- TensorFlow and PyTorch tensors preserve NumPy's array semantics while adding gradients, devices, and framework state.
- Token IDs are addresses; embedding lookup turns `(B,T)` into `(B,T,D)`.

**Out of scope in this chapter (and where to find it):**

- Autograd and explicit optimization - continue to [`optional/01-keras-to-pytorch-antarctic-field-guide.ipynb`](optional/01-keras-to-pytorch-antarctic-field-guide.ipynb).
- Padding, masked sequence loss, gradient clipping, and autoregressive generation - continue to [`../../pytorch-for-llms/05-sequence-memory/README.md`](../../pytorch-for-llms/05-sequence-memory/README.md).
- BPE vocabulary construction - see [`../../pytorch-for-llms/06-tokenization-and-embeddings/README.md`](../../pytorch-for-llms/06-tokenization-and-embeddings/README.md).
- Q/K/V and multi-head attention - see [`../../genai/01-transformers/02-attention-and-position.ipynb`](../../genai/01-transformers/02-attention-and-position.ipynb).

**Forward: what you just built will be used immediately.**

The Antarctic Field Guide starts from a real `(examples, features)` table. Its first tensor conversion, dense layer, and explicit training loop reuse the axis, dtype, and weight-orientation contracts you proved here.

> **Next:** [`optional/01-keras-to-pytorch-antarctic-field-guide.ipynb`](optional/01-keras-to-pytorch-antarctic-field-guide.ipynb)